In [14]:
import requests, os
import json
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://www.bseindia.com/"
}

path = r"Book3.xlsx"

df = pd.read_excel(path)
df.head(5)

,Company_name,isin,file_name,url
0,20 Microns Ltd.,INE144J01027,/Notices/31Jul2026/9f1f58a7-2e42-4800-9ee2-94...,https://www.bseindia.com/xml-data/corpfiling/...
1,360 One Wam Ltd.,INE466L01038,/Notices/16Jul2026/5a36c94c-f99e-42f3-88f7-7c...,https://www.bseindia.com/xml-data/corpfiling/...
2,3B Blackbio Dx Ltd.,INE994E01018,/Notices/13Aug2026/c48a7b33-54ee-4c32-9a9f-b4...,
3,3I Infotech Ltd.,INE748C01038,/Notices/23Jul2026/693d2b52-5781-4be8-b1a4-6b...,
4,3M India Ltd.,INE470A01017,/Notices/14Aug2026/8dd22ca9-4970-44a2-9223-b9...,


In [ ]:
import pandas as pd
import requests
from pathlib import Path

df = pd.read_excel(path)
output_dir = Path(r"D:\Q1_2026_PDFS")
output_dir.mkdir(exist_ok=True)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.bseindia.com/"
}


session = requests.Session()
session.headers.update(headers)

for _, row in df.iterrows():

    url = row[" url"]
    if not url or pd.isna(url):
        continue

    filename = str(row["Company_name"]).strip()
    if not filename.lower().endswith(".pdf"):
        filename += ".pdf"

    file_path = output_dir / filename
    try:
        print(f"Downloading: {filename}")

        response = session.get(
            url,
            stream=True,
            timeout=60,
            verify=False
        )

        response.raise_for_status()

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        print(f"Done: {filename}")

    except Exception as e:
        print(f"Failed: {filename} -> {e}")

In [ ]:
import requests
import time
import urllib3
from datetime import datetime, timedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.bseindia.com/",
    "Origin": "https://www.bseindia.com",
}

url = "https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w"

session = requests.Session()
session.headers.update(headers)

# Establish BSE session
session.get(
    "https://www.bseindia.com/",
    timeout=10,
    verify=False
)

today = datetime.now()
total_set = 90

report_list = []

while total_set:

    today_dt = today.strftime("%Y%m%d")
    prev = today - timedelta(days=1)
    prev_dt = prev.strftime("%Y%m%d")

    params = {
        "pageno": 1,
        "strCat": "Result",
        "strPrevDate": prev_dt,
        "strScrip": "",
        "strSearch": "P",
        "strToDate": today_dt,
        "strType": "C",
        "subcategory": -1
    }

    print(f"Fetching: {prev_dt} -> {today_dt}")

    try:

        response = session.get(
            url,
            params=params,
            timeout=20,
            verify=False
        )

        print("Status:", response.status_code)

        if response.status_code == 200:

            api_data = response.json()

            data = api_data.get("Table", [])

            print("Records:", len(data))

            if data:
                report_list.extend(data)

        else:
            print("Error:", response.text[:500])

    except requests.exceptions.RequestException as error:
        print("Network Connection Failed:", error)

    except ValueError:
        print("Invalid JSON response")
        print(response.text[:500])

    # Move window back by 30 days
    today = prev
    total_set -= 1

    time.sleep(1)

print(f"\nTotal reports collected: {len(report_list)}")

##REPORTS FOR 2025 and 2026

In [ ]:
# import os
# import requests

# DOWNLOAD_FOLDER = r"E:\ANNUAL_REPORTS_2024"
# s = requests.Session()
# s.headers.update(headers)
# s.get("https://nseindia.com")

# os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

# for idx, row in df26.iterrows():
#     url = row["fileName"]
#     filename = f"{DOWNLOAD_FOLDER}/{row["symbol"]}.pdf"

#     print(f"Downloading {filename}...")
#     res = s.get(url)
#     with open(filename, "wb") as f:
#         f.write(res.content)

#     if not res.content.startswith(b"%PDF"):
#         print("Corrupt or blocked. Deleting...")
#         os.remove(filename)
#     else:
#         print("Saved!")
#     time.sleep(.5)
